# Writable Postgres tables — execution evidence

Proves the operational tables are **writable** and **distinct** from the read-only synced table `public.store_sku_position_synced`. We INSERT and UPDATE a recovery action (the audit trigger fires), then contrast with the synced table.

In [1]:
import sys, textwrap
sys.path.insert(0, ".")
from lib import lb
conn = lb.connect("geniebandits-dev")
conn.autocommit = True
cur = conn.cursor()

def show(sql, title=None):
    if title: print(f"### {title}")
    print(textwrap.dedent(sql).strip())
    cur.execute(sql)
    if cur.description:
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
        print("-> " + " | ".join(cols))
        for r in rows:
            print("   " + " | ".join(str(x) for x in r))
        print(f"({len(rows)} row(s))\n")
    else:
        print(f"-> OK ({cur.rowcount} affected)\n")

In [2]:
cur.execute("""INSERT INTO northpeak_ops.recovery_actions
  (store_id, product_id, chosen_move, units, net_recaptured_value, status, approved_by)
  VALUES ('STORE-0003','SKU-APP-04414','expedite',15,9100.0,'pending','evidence.nb')
  RETURNING action_id""")
aid = cur.fetchone()[0]
print("inserted recovery_actions.action_id =", aid)

inserted recovery_actions.action_id = 7


In [3]:
cur.execute("UPDATE northpeak_ops.recovery_actions SET status='approved', approved_by='evidence.nb' WHERE action_id=%s", (aid,))
print("updated action", aid, "-> approved (writable UPDATE succeeded)")

updated action 7 -> approved (writable UPDATE succeeded)


In [4]:
show(f"""SELECT action_id, store_id, product_id, chosen_move, status, approved_by
        FROM northpeak_ops.recovery_actions WHERE action_id={aid}""",
     "The writable row")

### The writable row
SELECT action_id, store_id, product_id, chosen_move, status, approved_by
        FROM northpeak_ops.recovery_actions WHERE action_id=7


-> action_id | store_id | product_id | chosen_move | status | approved_by
   7 | STORE-0003 | SKU-APP-04414 | expedite | approved | evidence.nb
(1 row(s))



In [5]:
show(f"""SELECT old_status, new_status, changed_by
        FROM northpeak_ops.action_status_history WHERE action_id={aid}""",
     "Audit history written by the trigger")

### Audit history written by the trigger
SELECT old_status, new_status, changed_by
        FROM northpeak_ops.action_status_history WHERE action_id=7


-> old_status | new_status | changed_by
   pending | approved | evidence.nb
(1 row(s))



In [6]:
show("""SELECT table_schema, table_name FROM information_schema.tables
        WHERE (table_schema='northpeak_ops' AND table_name='recovery_actions')
           OR (table_schema='public' AND table_name='store_sku_position_synced')
        ORDER BY 1""", "Writable ops table vs read-only synced table (distinct objects)")

### Writable ops table vs read-only synced table (distinct objects)
SELECT table_schema, table_name FROM information_schema.tables
        WHERE (table_schema='northpeak_ops' AND table_name='recovery_actions')
           OR (table_schema='public' AND table_name='store_sku_position_synced')
        ORDER BY 1


-> table_schema | table_name
   northpeak_ops | recovery_actions
   public | store_sku_position_synced
(2 row(s))



In [7]:
show("SELECT count(*) AS synced_rows FROM public.store_sku_position_synced",
     "Synced table row count (pipeline-managed mirror)")

### Synced table row count (pipeline-managed mirror)
SELECT count(*) AS synced_rows FROM public.store_sku_position_synced


-> synced_rows
   14000
(1 row(s))

